In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

In [2]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
start = NY_tz.localize(datetime.datetime(2025, 12, 29, 7, 0))
end = NY_tz.localize(datetime.datetime(2025, 12, 29, 17, 00))

# from SDRUtils.data.builder import SDRDataBuilder
# sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
# df = sdr.grab_sdr_trades(
# 	start_timestamp=start,
# 	end_timestamp=end,
# 	agency="CFTC",
# 	asset_class="RATES",
# )
# df

In [3]:
from SDRUtils.products import USD_SOFR_SwapProduct

product = USD_SOFR_SwapProduct()
cdf = product.build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=True)
cdf

FETCHING DELIVERY BASKETS...: 100%|██████████| 6/6 [00:07<00:00,  1.20s/it]


,trade_id,execution_timestamp,effective_date,expiration_date,product_type,tenor_years,tenor_label,is_forward,forward_start_years,forward_label,...,ust_cusip,ust_oi,ust_issue_date,swap_maturity_date,matched_ust_maturity_trade_confidence,invoice_swap_ticker,is_mac,is_spreadover,is_asset_swap,risk
0,1570345282000003401,2024-01-23 20:03:22+00:00,2024-01-25,2031-01-25 00:00:00,UNKNOWN,7.108333,7Y,False,0.005556,spot,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,600.0
1,1570758957000000101,2024-07-11 15:06:00+00:00,2024-07-31,2038-03-15 00:00:00,OIS_SWAP,13.819444,IMM_H2038,True,0.055556,FOMC_20240731,...,NaN,NaN,NaN,2038-03-15,NaN,NaN,False,False,False,53000.0
2,1570686146000000601,2024-07-15 16:08:43+00:00,2024-08-05,2038-03-15 00:00:00,OIS_SWAP,13.805556,IMM_H2038,True,0.058333,3W,...,NaN,NaN,NaN,2038-03-15,NaN,NaN,False,False,False,43500.0
3,1570760664000000101,2024-08-01 15:05:00+00:00,2024-08-22,2038-03-15 00:00:00,OIS_SWAP,13.758333,IMM_H2038,True,0.058333,3W,...,NaN,NaN,NaN,2038-03-15,NaN,NaN,False,False,False,43500.0
4,1570346918000001301,2024-07-30 17:18:59+00:00,2024-08-30,2027-12-31 00:00:00,UNKNOWN,3.383333,3Y,True,0.086111,1M,...,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,29400.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1955,1573637159000000501 / 1573637158000000401,2025-12-29T21:40:22+00:00 / 2025-12-29T21:40:1...,2026-01-07,2026-12-07T00:00:00 / 2027-01-06T00:00:00,OIS_SWAP,0.927778 / 1.01111,11M / 1Y,True,0.025000,1W,...,NaN,NaN,NaN,2026-12-07 / 2027-01-06,NaN,NaN,False,False,False,300.0
1956,1573649665000000201,2025-12-29 21:53:50+00:00,2025-12-31,2029-12-31 00:00:00,OIS_SWAP,4.058333,4Y,False,0.005556,spot,...,91282CMD0,5-Year,2024-12-31,2029-12-31,low,NaN,False,False,False,1500.0
1957,1573656731000000101,2025-12-29 21:54:42+00:00,2025-12-31,2026-12-31 00:00:00,OIS_SWAP,1.013889,1Y,False,0.005556,spot,...,91282CME8,2-Year,2024-12-31,2026-12-31,low,NaN,False,False,False,4000.0
1958,1573666981000000101 / 1573660614000000101,2025-12-29T21:55:48+00:00 / 2025-12-29T21:55:3...,2025-12-31,2035-12-31T00:00:00 / 2055-12-31T00:00:00,OIS_SWAP,10.1444 / 30.4361,10Y / 30Y,False,0.005556,spot,...,NaN,NaN,NaN,2035-12-31 / 2055-12-31,NaN,NaN,False,False,False,41900.0


In [6]:
# cdf[cdf["product_type"] == "UNKNOWN"].to_dict(orient="records")
# df[df["Dissemination Identifier"] == 1573686006000000101].iloc[0].to_dict()

In [7]:
# cdf[cdf["trade_id"] == 1573577765000000201].iloc[0].to_dict(), cdf[cdf["trade_id"] == 1573612074000000101].iloc[0].to_dict()
# cdf[(cdf["package_legs"].isna()) & (cdf["Package indicator"] == True) & (cdf["Package transaction spread"].notna()) & (cdf["forward_label"] == "spot")]
# cdf[cdf["is_spreadover"] == True].iloc[0:5].to_dict(orient="records")